In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/diabetic_data_All_cleaned.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (101766, 41)


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,...,tolazamide,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted_30
0,Caucasian,Female,[0-10),6,25,1,1,41,0,1,...,No,No,No,No,No,No,No,No,No,0
1,Caucasian,Female,[10-20),1,1,7,3,59,0,18,...,No,Up,No,No,No,No,No,Ch,Yes,0
2,AfricanAmerican,Female,[20-30),1,1,7,2,11,5,13,...,No,No,No,No,No,No,No,No,Yes,0
3,Caucasian,Male,[30-40),1,1,7,2,44,1,16,...,No,Up,No,No,No,No,No,Ch,Yes,0
4,Caucasian,Male,[40-50),1,1,7,1,51,0,8,...,No,Steady,No,No,No,No,No,Ch,Yes,0


In [3]:
X = df.drop(columns=["readmitted_30"])
y = df["readmitted_30"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (101766, 40)
y shape: (101766,)


In [4]:
print("Target distribution:")
print(y.value_counts())

print("\nTarget percentage:")
print(y.value_counts(normalize=True) * 100)

Target distribution:
readmitted_30
0    90409
1    11357
Name: count, dtype: int64

Target percentage:
readmitted_30
0    88.840084
1    11.159916
Name: proportion, dtype: float64


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (81412, 40)
X_test shape: (20354, 40)
y_train shape: (81412,)
y_test shape: (20354,)


In [6]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Number of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

print("\nNumerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Number of numerical features: 11
Number of categorical features: 29

Numerical features:
['admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Categorical features:
['race', 'gender', 'age', 'diag_1', 'diag_2', 'diag_3', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


In [7]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [8]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

In [9]:
categorical_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [12]:
preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


In [13]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("X_train processed shape:", X_train_processed.shape)
print("X_test processed shape:", X_test_processed.shape)

X_train processed shape: (81412, 2271)
X_test processed shape: (20354, 2271)


In [15]:
print("Processed data type:", type(X_train_processed))
print("Number of processed features:", X_train_processed.shape[1])

Processed data type: <class 'scipy.sparse._csr.csr_matrix'>
Number of processed features: 2271


In [16]:
X_train_processed

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3256480 stored elements and shape (81412, 2271)>

In [17]:
X_test_processed

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 814056 stored elements and shape (20354, 2271)>

In [19]:
feature_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed.toarray(),
    columns=feature_names,
    index=X_train.index
)

print(X_train_processed_df.head())

       num__admission_type_id  num__discharge_disposition_id  \
46943               -0.706932                       3.645771   
60886               -0.706932                      -0.514411   
34417               -0.014635                      -0.514411   
57371               -0.706932                      -0.514411   
19626               -0.706932                      -0.136213   

       num__admission_source_id  num__time_in_hospital  \
46943                 -0.187628               2.211912   
60886                  0.305517              -1.135963   
34417                 -1.173919               0.203187   
57371                  0.305517              -1.135963   
19626                 -0.187628               1.877125   

       num__num_lab_procedures  num__num_procedures  num__num_medications  \
46943                 0.096863            -0.198173              0.243726   
60886                -0.563264             1.563785             -0.125745   
34417                 0.147642     

In [20]:
preview = pd.DataFrame(
    X_train_processed[:5].toarray(),
    columns=feature_names
)

print(preview)

   num__admission_type_id  num__discharge_disposition_id  \
0               -0.706932                       3.645771   
1               -0.706932                      -0.514411   
2               -0.014635                      -0.514411   
3               -0.706932                      -0.514411   
4               -0.706932                      -0.136213   

   num__admission_source_id  num__time_in_hospital  num__num_lab_procedures  \
0                 -0.187628               2.211912                 0.096863   
1                  0.305517              -1.135963                -0.563264   
2                 -1.173919               0.203187                 0.147642   
3                  0.305517              -1.135963                -0.360148   
4                 -0.187628               1.877125                 0.553874   

   num__num_procedures  num__num_medications  num__number_outpatient  \
0            -0.198173              0.243726               -0.291734   
1             1.5637

In [21]:
import joblib

joblib.dump(
    preprocessor,
    "../models/preprocessor.pkl"
)

print("Preprocessor saved successfully!")

Preprocessor saved successfully!


In [22]:
print("========== PREPROCESSING SUMMARY ==========")

print("Original features:", X.shape[1])
print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Processed features:", X_train_processed.shape[1])

print("\nTraining samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

print("\nPreprocessing completed successfully!")

========== PREPROCESSING SUMMARY ==========
Original features: 40
Numerical features: 11
Categorical features: 29
Processed features: 2271

Training samples: 81412
Testing samples: 20354

Preprocessing completed successfully!
